In [61]:
import polars as pl
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
DATASET_PATH = "/group/pmc021/amunif/epi-thesis/workflow/12_E066 with 5 common histones/dataset"

In [4]:
def load_histone(filename):
    # Create a schema
    intersect_schema = pl.Schema({
        # The E066.bed files
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.String,
        'label': pl.String,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64,

        # The gappedPeak column
        "chrom_p": pl.String,
        "chromStart_p": pl.Int64, 
        "chromEnd_p": pl.Int64, 
        "name_p": pl.String, 
        "score_p": pl.Float64, 
        "strand_p": pl.String,
        "thickStart": pl.Int64,
        "thickEnd": pl.Int64,
        "itemRgb": pl.Int64,
        "blockCount": pl.Int64,
        "blockSizes": pl.String,
        "blockStarts": pl.String,
        "signalValue": pl.Float64,
        "pValue": pl.Float64,
        "qValue": pl.Float64
    })

    # Open file
    histone_df = pl.read_csv(
            filename,
            separator="\t",
            has_header = False,
            schema = intersect_schema   
        )
    
    return histone_df

In [13]:
def load_gene_expression(filename):
    # Create a schema
    E066_schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.Int64,
        'label': pl.String,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
    })

    # Read the file
    df = pl.read_csv(filename, has_header=False, schema=E066_schema, separator="\t")

    return df
    

In [7]:
def create_empty_dataframe(histone_name):
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

In [16]:
def build_matrix(genes_df, histone_df, histone_name):
    # Build the dataframe with window
    genes_with_windows = genes_df.with_columns([
        pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_start')
    ]).explode('window_start')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_start') + 100).alias('window_end')
    ])

    # Join genes with histone data
    joined_df = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result_df = joined_df.filter(
        (pl.col('chromStart_p') < pl.col('window_end')) &
        (pl.col('chromEnd_p') > pl.col('window_start'))
    ).group_by(['gene_id', 'window_start'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_start'])

    # Find the gene without histone match
    genes_wo_histone = genes_with_windows.join(
        result_df,
        on=["gene_id", "window_start"],
        how="anti"
    )

    # Add the signalValue column so it can be merged
    genes_wo_histone = genes_wo_histone.with_columns(
        signalValue = pl.lit(0.0).cast(pl.Float64)
    )

    # Aggregate the genes without histone result
    genes_wo_histone = genes_wo_histone.group_by(['gene_id', 'window_start'], maintain_order=True).agg([
            pl.col('signalValue').mean().alias(histone_name)
        ]).sort(['gene_id', 'window_start'])

    # Merge both (results and genes without histone)
    result_df.extend(genes_wo_histone)

    # Sort the result dataframe by gene_id and window start for aggregation
    sorted_result_df = result_df.sort(['gene_id', 'window_start'])
    
    # Group by to make array of features
    matrix_df = (
        sorted_result_df
        .with_columns(pl.col(histone_name).fill_null(0))
        .group_by(['gene_id'], maintain_order=True)
        .agg(pl.col(histone_name))
        .sort('gene_id')
    )

    # Add the count and length of array feature for checking
    matrix_control_df = matrix_df.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    matrix_control_df = matrix_control_df.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    # Finally, return the gene_id with its histone features
    return matrix_control_df

In [17]:
# Getting histone in chunk
def get_histone_features(genes_df, histone_df, histone_name):
    
    genes_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes_df.iter_slices(n_rows=100)):
        if i % 100 == 0:
            print(f"Processing {histone_name}: {i*100}/{genes_df.height}")
        
        result = build_matrix(chunk, histone_df, histone_name)
        genes_w_histone.extend(result)

    print(f"Processing {histone_name} features finished.")
    return genes_w_histone

In [19]:
# Load the E066 file
genes_df = load_gene_expression(os.path.join(DATASET_PATH, "E066.bed"))

In [20]:
genes_df

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,str,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,"""1""","""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,"""0""","""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,"""1""","""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,"""1""","""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,"""0""","""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,"""0""","""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,"""0""","""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,"""0""","""RP11-812E19.9""",33647044,33647696,33647696


In [5]:
# Load the histone dataset
H3K9ac_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K9ac.bed'))
H3K9me3_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K9me3.bed'))
H3K4me3_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K4me3.bed'))
H3K27ac_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K27ac.bed'))
H3K27me3_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K27me3.bed'))

In [6]:
H3K9ac_df

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,chrom_p,chromStart_p,chromEnd_p,name_p,score_p,strand_p,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts,signalValue,pValue,qValue
str,i64,i64,str,f64,str,str,str,i64,i64,i64,str,i64,i64,str,f64,str,i64,i64,i64,i64,str,str,f64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,"""-1""","""1""","""TSPAN6""",99883667,99894988,99894988,"""chrX""",99888701,99892952,"""Rank_11507""",27.0,""".""",99890439,99892429,0,3,"""221,918,465""","""1738,2172,3263""",3.18521,4.55219,2.71766
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,"""-1""","""1""","""TSPAN6""",99883667,99894988,99894988,"""chrX""",99899209,99909708,"""Rank_50104""",3.0,""".""",99904843,99905132,0,1,"""289""","""5634""",1.92092,1.80065,0.38807
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,"""-1""","""1""","""DPM1""",49551404,49575092,49575092,"""chr20""",49569539,49576775,"""Rank_1898""",97.0,""".""",49571056,49576238,0,2,"""238,2795""","""1517,3904""",4.46174,11.87399,9.78464
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,"""-1""","""1""","""SCYL3""",169818772,169863408,169863408,"""chr1""",169859501,169864111,"""Rank_3934""",70.0,""".""",169860030,169864102,0,4,"""1173,217,975,1185""","""529,1857,2335,3416""",3.79153,9.17427,7.08451
"""chr1""",27956788,27966788,"""ENSG00000000938""",4.572,"""-1""","""1""","""FGR""",27938575,27961788,27961788,"""chr1""",27955001,27962268,"""Rank_25087""",10.0,""".""",27955828,27961238,0,5,"""317,197,205,321,372""","""827,3098,3959,5433,5865""",2.23555,2.5748,1.02445
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",66869528,66879528,"""ENSG00000259471""",0.412,"""1""","""0""","""RP11-321F6.1""",66874528,66978132,66874528,"""chr15""",66873913,66876438,"""Rank_43762""",4.0,""".""",66874604,66874891,0,1,"""287""","""691""",2.0501,1.96626,0.49456
"""chr15""",71036271,71046271,"""ENSG00000259532""",0.0,"""1""","""0""","""RP11-138H8.2""",71041271,71046516,71041271,"""chr15""",71031650,71037438,"""Rank_9058""",36.0,""".""",71033443,71037078,0,3,"""850,250,1644""","""1793,2797,3784""",3.27377,5.56016,3.67582
"""chr15""",80210113,80220113,"""ENSG00000259642""",1.753,"""1""","""0""","""C15orf37""",80215113,80217194,80215113,"""chr15""",80213780,80217088,"""Rank_5547""",56.0,""".""",80214846,80216823,0,2,"""1265,501""","""1066,2542""",3.82821,7.72815,5.66485


In [23]:
# Build the matrix
genes_w_H3K9ac_df = get_histone_features(genes_df, H3K9ac_df, 'H3K9ac')
genes_w_H3K9me3_df = get_histone_features(genes_df, H3K9me3_df, 'H3K9me3')
genes_w_H3K4me3_df = get_histone_features(genes_df, H3K4me3_df, 'H3K4me3')
genes_w_H3K27ac_df = get_histone_features(genes_df, H3K27ac_df, 'H3K27ac')
genes_w_H3K27me3_df = get_histone_features(genes_df, H3K27me3_df, 'H3K27me3')

Processing H3K9ac: 0/19645
Processing H3K9ac: 10000/19645
Processing H3K9ac features finished.
Processing H3K9me3: 0/19645
Processing H3K9me3: 10000/19645
Processing H3K9me3 features finished.
Processing H3K4me3: 0/19645
Processing H3K4me3: 10000/19645
Processing H3K4me3 features finished.
Processing H3K27ac: 0/19645
Processing H3K27ac: 10000/19645
Processing H3K27ac features finished.
Processing H3K27me3: 0/19645
Processing H3K27me3: 10000/19645
Processing H3K27me3 features finished.


# Checking the generated features

In [24]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_wc') > 0).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32
"""ENSG00000003989""","[5.46943, 5.46943, … 5.46943]",100,100
"""ENSG00000005483""","[5.77316, 5.77316, … 5.77316]",100,100
"""ENSG00000005889""","[3.94027, 3.94027, … 3.94027]",100,100
"""ENSG00000011260""","[3.51395, 3.51395, … 3.51395]",100,100
"""ENSG00000023839""","[3.8014, 3.8014, … 3.8014]",100,100
…,…,…,…
"""ENSG00000175877""","[2.13207, 0.0, … 0.0]",1,100
"""ENSG00000176092""","[0.0, 0.0, … 2.89959]",1,100
"""ENSG00000213953""","[4.84417, 0.0, … 0.0]",1,100


In [25]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_len') < 100).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32


In [26]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_wc') > 0).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""ENSG00000062370""","[2.92435, 2.92435, … 2.92435]",100,100
"""ENSG00000081665""","[3.19758, 3.19758, … 3.19758]",100,100
"""ENSG00000081818""","[2.56639, 2.56639, … 2.56639]",100,100
"""ENSG00000081842""","[3.40924, 3.40924, … 3.40924]",100,100
"""ENSG00000081853""","[3.79613, 3.79613, … 3.79613]",100,100
…,…,…,…
"""ENSG00000206026""","[2.44841, 0.0, … 0.0]",1,100
"""ENSG00000206536""","[0.0, 0.0, … 2.61637]",1,100
"""ENSG00000213401""","[0.0, 0.0, … 1.87772]",1,100


In [27]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_len') < 100).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32


In [28]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000002933""","[7.03481, 7.03481, … 7.03481]",100,100
"""ENSG00000005421""","[6.19671, 6.19671, … 6.19671]",100,100
"""ENSG00000005483""","[6.56627, 6.56627, … 6.56627]",100,100
"""ENSG00000018869""","[3.29138, 3.29138, … 3.29138]",100,100
"""ENSG00000023839""","[4.90903, 4.90903, … 4.90903]",100,100
…,…,…,…
"""ENSG00000221855""","[2.15356, 0.0, … 0.0]",1,100
"""ENSG00000230301""","[0.0, 0.0, … 1.96278]",1,100
"""ENSG00000237330""","[3.87111, 0.0, … 0.0]",1,100


In [29]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000002933""","[7.03481, 7.03481, … 7.03481]",100,100
"""ENSG00000005421""","[6.19671, 6.19671, … 6.19671]",100,100
"""ENSG00000005483""","[6.56627, 6.56627, … 6.56627]",100,100
"""ENSG00000018869""","[3.29138, 3.29138, … 3.29138]",100,100
"""ENSG00000023839""","[4.90903, 4.90903, … 4.90903]",100,100
…,…,…,…
"""ENSG00000221855""","[2.15356, 0.0, … 0.0]",1,100
"""ENSG00000230301""","[0.0, 0.0, … 1.96278]",1,100
"""ENSG00000237330""","[3.87111, 0.0, … 0.0]",1,100


In [30]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_len') < 100).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32


In [31]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_wc') > 0).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32
"""ENSG00000002933""","[6.09209, 6.09209, … 6.09209]",100,100
"""ENSG00000003989""","[5.87857, 5.87857, … 5.87857]",100,100
"""ENSG00000004399""","[5.01725, 5.01725, … 5.01725]",100,100
"""ENSG00000005421""","[3.94137, 3.94137, … 3.94137]",100,100
"""ENSG00000005483""","[7.31397, 7.31397, … 7.31397]",100,100
…,…,…,…
"""ENSG00000196188""","[2.13414, 0.0, … 0.0]",1,100
"""ENSG00000196208""","[0.0, 0.0, … 2.08693]",1,100
"""ENSG00000198483""","[5.92595, 0.0, … 0.0]",1,100


In [32]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_len') < 100).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32


In [33]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_wc') > 0).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""ENSG00000002746""","[2.4902, 2.4902, … 2.4902]",100,100
"""ENSG00000005073""","[3.08264, 3.08264, … 3.08264]",100,100
"""ENSG00000006377""","[3.1973, 3.1973, … 3.1973]",100,100
"""ENSG00000007372""","[3.27537, 3.27537, … 3.27537]",100,100
"""ENSG00000009709""","[2.82916, 2.82916, … 2.82916]",100,100
…,…,…,…
"""ENSG00000196156""","[0.0, 0.0, … 2.64951]",1,100
"""ENSG00000196966""","[0.0, 0.0, … 2.48854]",1,100
"""ENSG00000203780""","[0.0, 0.0, … 2.09997]",1,100


In [34]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_len') < 100).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32


# Join all histones into single dataframe

In [35]:
# Join all histones into single dataframe
genes_histone_df = genes_w_H3K9ac_df \
                    .join(genes_w_H3K9me3_df, on='gene_id') \
                    .join(genes_w_H3K4me3_df, on='gene_id') \
                    .join(genes_w_H3K27ac_df, on='gene_id') \
                    .join(genes_w_H3K27me3_df, on='gene_id')

In [36]:
genes_histone_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""ENSG00000000003""","[3.18521, 3.18521, … 1.92092]",38,100,"[0.0, 0.0, … 0.0]",0,100,"[4.89323, 4.89323, … 0.0]",31,100,"[3.84344, 3.84344, … 5.45294]",41,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",20,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 2.29406]",3,100
"""ENSG00000000419""","[4.46174, 4.46174, … 0.0]",67,100,"[0.0, 0.0, … 0.0]",8,100,"[0.0, 0.0, … 0.0]",47,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",48,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",41,100,"[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[1.79162, 1.79162, … 0.0]",29,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",20,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


# Join genes with the value and histone features

In [38]:
# Join genes with the value and histone features
genes_values_df = genes_df.select(['gene_id', 'E066'])
genes_values_df

gene_id,E066
str,f64
"""ENSG00000000003""",73.205
"""ENSG00000000005""",0.191
"""ENSG00000000419""",52.609
"""ENSG00000000457""",4.733
"""ENSG00000000460""",0.942
…,…
"""ENSG00000259658""",0.212
"""ENSG00000259664""",0.0
"""ENSG00000259680""",0.071


In [39]:
genes_histone_values_df = genes_histone_df.join(
    genes_values_df,
    on = 'gene_id'
)

In [40]:
genes_histone_values_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[3.18521, 3.18521, … 1.92092]",38,100,"[0.0, 0.0, … 0.0]",0,100,"[4.89323, 4.89323, … 0.0]",31,100,"[3.84344, 3.84344, … 5.45294]",41,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",20,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 2.29406]",3,100,0.191
"""ENSG00000000419""","[4.46174, 4.46174, … 0.0]",67,100,"[0.0, 0.0, … 0.0]",8,100,"[0.0, 0.0, … 0.0]",47,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",48,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",41,100,"[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[1.79162, 1.79162, … 0.0]",29,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",20,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071


In [41]:
# Save to parquet
genes_histone_values_df.write_parquet(os.path.join(DATASET_PATH, 'E066_exp_histones.parquet'))

In [42]:
test_df = pl.read_parquet(os.path.join(DATASET_PATH, 'E066_exp_histones.parquet'))
test_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[3.18521, 3.18521, … 1.92092]",38,100,"[0.0, 0.0, … 0.0]",0,100,"[4.89323, 4.89323, … 0.0]",31,100,"[3.84344, 3.84344, … 5.45294]",41,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",20,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 2.29406]",3,100,0.191
"""ENSG00000000419""","[4.46174, 4.46174, … 0.0]",67,100,"[0.0, 0.0, … 0.0]",8,100,"[0.0, 0.0, … 0.0]",47,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",48,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",41,100,"[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[1.79162, 1.79162, … 0.0]",29,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",20,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071


In [43]:
marker_df = pl.read_parquet(os.path.join(DATASET_PATH, "marker_combinations.parquet"))

In [44]:
marker_df

combination
list[str]
"[""H3K4me3""]"
"[""H3K9ac""]"
"[""H3K9me3""]"
"[""H3K27ac""]"
"[""H3K27me3""]"
…
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9me3"", … ""H3K27me3""]"


# Prepare the index for training, validation, and testing

In [49]:
# Convert to numpy array
all_features_np = genes_histone_values_df.to_numpy()
print(all_features_np)
print(all_features_np.shape)

[['ENSG00000000003'
  array([3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521,
         3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521,
         3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521,
         3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521, 3.18521,
         3.18521, 3.18521, 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
         0.     , 1.92092, 1.92092, 1.92092,

In [52]:
# Split train, test, validation by index
data_indices = np.arange(len(all_features_np))
print(data_indices)

[    0     1     2 ... 19642 19643 19644]


In [56]:
# First split: 80% train, 20% temporary (for test + validation)
train_idx, temp_idx = train_test_split(
    data_indices, 
    test_size=0.2, 
    random_state=42  # For reproducibility
)

In [57]:
# Second split: Split temp_idx into 50% test and 50% validation
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.5, 
    random_state=42  # Same random_state for consistency
)

In [59]:
print(train_idx.shape)
print(val_idx.shape)
print(test_idx.shape)

(15716,)
(1964,)
(1965,)


In [62]:
# Save train, val, and test into parquet file
train_idx_df = pd.DataFrame(train_idx, columns=['values'])
train_idx_df.to_parquet(os.path.join(DATASET_PATH, 'train_idx.parquet'))

val_idx_df = pd.DataFrame(val_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_PATH, 'val_idx.parquet'))

test_idx_df = pd.DataFrame(test_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_PATH, 'test_idx.parquet'))